<a href="https://colab.research.google.com/github/danielsiso461/Lab2/blob/master/Firebase_WordCounter_Colab_Ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install firebase

In [ ]:

import requests
import re
from collections import Counter
from urllib.parse import quote

URL = "https://webproj-ba2c4-default-rtdb.firebaseio.com/wordCounts"

def word_url(word):
    return f"{URL}/{quote(word, safe='')}.json"

def read(url):
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    return response.json()

def write(url, value):
    response = requests.put(url, json=value, timeout=15)
    response.raise_for_status()

def split_words(text):
    return re.findall(r"[^\W_]+", text.lower())

def add_word(word, amount=1):
    url = word_url(word)
    current = read(url) or 0
    write(url, current + amount)

while True:
    print("\n--- Word Counter Menu ---")
    print("1. Add a word")
    print("2. Add text")
    print("3. Update word count")
    print("4. Delete word")
    print("5. View all words")
    print("6. Exit")

    choice = input("Choose 1-6: ")

    try:
        if choice == "1":
            words = split_words(input("Enter word: "))
            if len(words) == 1:
                add_word(words[0])
                print("Word added!")
            else:
                print("Enter exactly one word.")

        elif choice == "2":
            text = input("Enter text: ")
            counts = Counter(split_words(text))
            for word, amount in counts.items():
                add_word(word, amount)
            print("Text added!")

        elif choice == "3":
            words = split_words(input("Enter word: "))
            if len(words) != 1:
                print("Enter exactly one word.")
                continue

            url = word_url(words[0])
            if read(url) is None:
                print("Word not found.")
                continue

            try:
                count = int(input("New count: "))
                if count < 0:
                    raise ValueError
            except ValueError:
                print("Enter a non-negative integer.")
                continue

            write(url, count)
            print("Updated!")

        elif choice == "4":
            words = split_words(input("Enter word: "))
            if len(words) != 1:
                print("Enter exactly one word.")
                continue

            response = requests.delete(word_url(words[0]), timeout=15)
            response.raise_for_status()
            print("Deleted!")

        elif choice == "5":
            records = read(URL + ".json") or {}
            if not records:
                print("No words found.")
            else:
                for word, count in sorted(records.items()):
                    print(f"{word}: {count}")

        elif choice == "6":
            print("Goodbye!")
            break

        else:
            print("Invalid choice.")

    except requests.RequestException as error:
        print("Firebase error:", error)



--- Word Counter Menu ---
1. Add a word
2. Add text
3. Update word count
4. Delete word
5. View all words
6. Exit
Choose 1-6: 1
Enter word: hey
Word added!

--- Word Counter Menu ---
1. Add a word
2. Add text
3. Update word count
4. Delete word
5. View all words
6. Exit
Choose 1-6: 2
Enter text: whats up
Text added!

--- Word Counter Menu ---
1. Add a word
2. Add text
3. Update word count
4. Delete word
5. View all words
6. Exit
